# System Architecture


# Joke Retrieval and Recommendation System (Jester Dataset)

## 1. Problem statement
Given a user (the "query") and a collection of 100 jokes (the "documents"), the system returns a ranked
list of jokes the user is likely to enjoy. It can also answer keyword and Boolean searches over the joke text.

## 2. Dataset
Jester Collaborative Filtering Dataset (Goldberg et al., 2001):
- `JokeText.csv` – 100 jokes (text)
- `UserRatings1.csv`, `UserRatings2.csv` – 73,421 users, 4,136,360 ratings from -10 to +10
- A joke is treated as **relevant** for a user if the user rated it **>= 5**.

## 3. System architecture

```
 JokeText.csv ---> [Text preprocessing] ---> [Inverted index] ---> Boolean search
                   tokenise, stop words,        term -> postings     (AND / OR / NOT)
                   stemming                          |
                                                     v
                                          [TF-IDF vector space] ---> keyword search (cosine)
                                                     |
 UserRatings*.csv ---> [Ratings preprocessing]       |
                       user x joke matrix,           |
                       train / hidden split          |
                            |                        |
                            v                        v
                  +-----------------------------------------------+
                  |  Retrieval models:  Popularity | Content-based |
                  |  Item-based CF | Hybrid (CF + content)        |
                  +-----------------------------------------------+
                                    |
                                    v
                  [Query processing]  user -> score all jokes -> drop seen -> sort -> top-k
                                    |
                                    v
                  [Evaluation]  P@k, Recall@k, MAP, MRR, nDCG@k
```

## 4. Methods used (from the course)
Boolean retrieval and inverted index (Weeks 1-2), TF-IDF and cosine similarity in the vector space
model (Week 3), evaluation metrics (Week 4).

## Step 1: Import libraries and set parameters

In [4]:
pip install numpy pandas scikit-learn matplotlib seaborn

  Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached pillow-12.3.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 6.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 15.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 18.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.5 MB/s  0:00:00
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached pillow-12.3.0-cp314-cp314-

In [5]:
import re
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.preprocessing import normalize

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

SEED = 42                 # makes the random split reproducible
rng = np.random.default_rng(SEED)

DATA_DIR = "data"            # folder that contains the three CSV files
THRESHOLD = 5.0           # a joke is "relevant" if the user rated it >= 5
TEST_FRACTION = 0.30      # share of each user's ratings hidden for evaluation
N_EVAL_USERS = 3000       # number of users used in the evaluation
MIN_CANDIDATES = 10       # a user needs >= 10 hidden ratings to be evaluated

ModuleNotFoundError: No module named 'nltk'